# 30-机器学习量化入门

> 模块 4.2 ML量化 | 从因子特征到第一个可验证的机器学习预测模型

## 学习目标

- 理解 ML 量化的最小闭环：特征 $X_t$ → 标签 $y_{t+1}$ → 模型 → 样本外评估
- 会构造未来收益率标签，避免前视偏差
- 会按时间顺序切分训练集和测试集
- 用 numpy/pandas 实现一个线性 baseline，不额外引入新依赖
- 用方向准确率、收益曲线、Sharpe、最大回撤评估模型

## 环境依赖

本 notebook 使用仓库已有依赖：`numpy`、`pandas`、`matplotlib`。

如果本地有价格数据，可以替换为真实数据；如果没有，代码会自动使用固定随机种子的模拟数据。


## 1. ML 量化的基本问题

上一课我们已经把原始行情加工成可建模的因子特征。第 30 课开始进入真正的机器学习建模。

量化里的监督学习可以写成：

$$ y_{t+h} = f(X_t) + \varepsilon_{t+h} $$

- $X_t$：在 t 时刻已经知道的特征，例如动量、波动率、成交量变化
- $y_{t+h}$：未来 h 天的收益或涨跌方向
- $f$：模型学到的映射关系

关键不是“模型越复杂越好”，而是：标签是否正确、切分是否按时间、评估是否样本外。


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["font.sans-serif"] = ["Arial Unicode MS", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


## 2. 数据准备：真实数据优先，模拟数据兜底

为了让 notebook 在离线环境也能跑通，下面的函数会先尝试读取本地 CSV；如果没有文件，就生成一段带有趋势、波动聚集和噪声的模拟价格序列。

这不是为了“假装实盘”，而是为了让学习流程稳定复现。真实研究时，应替换成可靠行情数据。


In [ ]:
def load_price_data(csv_path="data/index_price.csv", n=900):
    """读取本地价格数据；失败时使用模拟数据降级。

    期望 CSV 至少包含 close 列，可选 date 列。
    """
    try:
        df = pd.read_csv(csv_path)
        if "date" in df.columns:
            df["date"] = pd.to_datetime(df["date"])
            df = df.set_index("date")
        if "close" not in df.columns:
            raise ValueError("CSV must contain a close column")
        out = df[["close"]].dropna().copy()
        if len(out) < 260:
            raise ValueError("Need at least 260 rows for rolling features")
        print(f"使用本地数据: {csv_path}, rows={len(out)}")
        return out
    except Exception as exc:
        print(f"无法读取真实数据，使用模拟数据降级: {exc}")
        dates = pd.bdate_range("2021-01-01", periods=n)
        regime = np.sin(np.linspace(0, 8 * np.pi, n)) * 0.0008
        volatility = 0.009 + 0.006 * (np.sin(np.linspace(0, 5 * np.pi, n)) > 0)
        noise = np.random.normal(0, volatility, size=n)
        returns = 0.00025 + regime + noise
        close = 100 * np.exp(np.cumsum(returns))
        return pd.DataFrame({"close": close}, index=dates)

price = load_price_data()
price.head()


In [ ]:
price["close"].plot(figsize=(10, 4), title="价格序列")
plt.ylabel("close")
plt.show()


## 3. 构造特征 X：只使用当前时点已知信息

这里构造一组很小但有代表性的特征：

| 特征 | 含义 | 注意点 |
|------|------|--------|
| `ret_1d` | 过去 1 日收益 | t 时点已知 |
| `mom_5d` | 过去 5 日动量 | t 时点已知 |
| `mom_20d` | 过去 20 日动量 | t 时点已知 |
| `vol_20d` | 过去 20 日波动率 | t 时点已知 |
| `ma_gap_20d` | 价格相对 20 日均线偏离 | t 时点已知 |
| `downside_vol_20d` | 下行波动率 | 风控特征 |

不要把未来收益、未来最高价、未来最低价混进特征。那是前视偏差。


In [ ]:
def make_features(df):
    out = df.copy()
    out["ret_1d"] = out["close"].pct_change()
    out["mom_5d"] = out["close"].pct_change(5)
    out["mom_20d"] = out["close"].pct_change(20)
    out["vol_20d"] = out["ret_1d"].rolling(20).std()
    out["ma_20d"] = out["close"].rolling(20).mean()
    out["ma_gap_20d"] = out["close"] / out["ma_20d"] - 1
    downside = out["ret_1d"].where(out["ret_1d"] < 0, 0)
    out["downside_vol_20d"] = downside.rolling(20).std()
    return out

features = make_features(price)
feature_cols = ["ret_1d", "mom_5d", "mom_20d", "vol_20d", "ma_gap_20d", "downside_vol_20d"]
features[feature_cols].tail()


## 4. 构造标签 y：预测未来 5 日收益方向

标签是 ML 量化里最容易出错的地方。

本课用未来 5 日收益作为标签：

$$ r_{t,t+5} = \frac{P_{t+5}}{P_t} - 1 $$

方向标签为：

$$ y_t = 1(r_{t,t+5} > 0) $$

注意：标签可以来自未来，因为它是训练答案；但特征必须只来自当前和过去。


In [ ]:
HORIZON = 5

data = features.copy()
data["future_ret_5d"] = data["close"].shift(-HORIZON) / data["close"] - 1
data["target_up"] = (data["future_ret_5d"] > 0).astype(int)

model_data = data[feature_cols + ["future_ret_5d", "target_up"]].dropna().copy()
print(model_data.shape)
model_data.head()


## 5. 时间顺序切分：训练过去，测试未来

随机切分会把未来市场状态泄露进训练过程。量化建模必须保持时间顺序。

这里用前 70% 训练，后 30% 测试。


In [ ]:
split = int(len(model_data) * 0.7)
train = model_data.iloc[:split].copy()
test = model_data.iloc[split:].copy()

X_train = train[feature_cols]
y_train = train["future_ret_5d"]
X_test = test[feature_cols]
y_test = test["future_ret_5d"]

print("train:", train.index.min(), "→", train.index.max(), train.shape)
print("test :", test.index.min(), "→", test.index.max(), test.shape)


## 6. 只在训练集上标准化

标准化也会泄露信息。正确做法是：

1. 用训练集计算均值和标准差
2. 训练集 transform
3. 测试集只使用训练集参数 transform

测试集永远不能参与拟合预处理参数。


In [ ]:
def fit_standardizer(X):
    mean = X.mean()
    std = X.std().replace(0, 1)
    return mean, std

def transform_standardizer(X, mean, std):
    return (X - mean) / std

mean, std = fit_standardizer(X_train)
X_train_scaled = transform_standardizer(X_train, mean, std)
X_test_scaled = transform_standardizer(X_test, mean, std)

X_train_scaled.describe().round(3)


## 7. Baseline 模型：手写岭回归预测未来收益

为了不引入额外依赖，本课用 numpy 解一个带 L2 正则的线性模型：

$$ \hat{\beta} = (X^TX + \lambda I)^{-1}X^Ty $$

它不是最强模型，但非常适合作为 baseline：

- 可解释
- 稳定
- 训练快
- 容易发现数据泄漏或标签错误


In [ ]:
def fit_ridge_regression(X, y, alpha=1.0):
    X_np = np.asarray(X, dtype=float)
    y_np = np.asarray(y, dtype=float)
    X_design = np.column_stack([np.ones(len(X_np)), X_np])
    penalty = np.eye(X_design.shape[1]) * alpha
    penalty[0, 0] = 0  # 截距不惩罚
    beta = np.linalg.solve(X_design.T @ X_design + penalty, X_design.T @ y_np)
    return beta

def predict_linear(X, beta):
    X_np = np.asarray(X, dtype=float)
    X_design = np.column_stack([np.ones(len(X_np)), X_np])
    return X_design @ beta

beta = fit_ridge_regression(X_train_scaled, y_train, alpha=5.0)
test["pred_ret_5d"] = predict_linear(X_test_scaled, beta)
test[["future_ret_5d", "pred_ret_5d"]].head()


In [ ]:
coef = pd.Series(beta[1:], index=feature_cols).sort_values()
coef.plot(kind="barh", figsize=(8, 4), title="线性 baseline 的特征权重")
plt.axvline(0, color="black", linewidth=1)
plt.show()


## 8. 从预测到策略信号

最简单的信号规则：

- 预测未来收益 > 0：持有
- 预测未来收益 ≤ 0：空仓

为了避免过度交易，本课先不做复杂仓位管理。真实策略还要考虑手续费、滑点、换手率和容量。


In [ ]:
test["signal"] = (test["pred_ret_5d"] > 0).astype(int)
# 用下一日收益近似日频策略收益：今天根据特征产生信号，下一日获得收益
test["next_1d_ret"] = price["close"].pct_change().shift(-1).reindex(test.index)
test["strategy_ret"] = test["signal"] * test["next_1d_ret"]
test["benchmark_ret"] = test["next_1d_ret"]

test[["signal", "next_1d_ret", "strategy_ret", "benchmark_ret"]].dropna().head()


## 9. 评估：方向准确率 + 收益指标

ML 量化不能只看分类准确率。方向对了但收益很小，方向错了但亏损很大，都可能让策略失败。

本课同时看：

| 指标 | 含义 | 注意 |
|------|------|------|
| 方向准确率 | 预测涨跌方向是否正确 | 容易忽略赔率 |
| 累计收益 | 策略最终收益 | 受样本路径影响 |
| Sharpe | 单位波动收益 | 年化假设要一致 |
| 最大回撤 | 最大历史亏损 | 风控核心指标 |
| 持仓比例 | 信号触发频率 | 太高/太低都要警惕 |


In [ ]:
def max_drawdown(equity):
    running_max = equity.cummax()
    drawdown = equity / running_max - 1
    return drawdown.min()

def sharpe_ratio(ret, periods=252):
    ret = ret.dropna()
    if ret.std() == 0 or len(ret) == 0:
        return np.nan
    return np.sqrt(periods) * ret.mean() / ret.std()

eval_df = test.dropna(subset=["strategy_ret", "benchmark_ret", "future_ret_5d", "pred_ret_5d"]).copy()
eval_df["pred_up"] = eval_df["pred_ret_5d"] > 0
eval_df["actual_up"] = eval_df["future_ret_5d"] > 0

strategy_equity = (1 + eval_df["strategy_ret"]).cumprod()
benchmark_equity = (1 + eval_df["benchmark_ret"]).cumprod()

metrics = pd.Series({
    "方向准确率": (eval_df["pred_up"] == eval_df["actual_up"]).mean(),
    "策略累计收益": strategy_equity.iloc[-1] - 1,
    "基准累计收益": benchmark_equity.iloc[-1] - 1,
    "策略Sharpe": sharpe_ratio(eval_df["strategy_ret"]),
    "基准Sharpe": sharpe_ratio(eval_df["benchmark_ret"]),
    "策略最大回撤": max_drawdown(strategy_equity),
    "持仓比例": eval_df["signal"].mean(),
})
metrics.round(4)


In [ ]:
pd.DataFrame({
    "strategy": strategy_equity,
    "benchmark": benchmark_equity,
}).plot(figsize=(10, 4), title="样本外累计收益：策略 vs 基准")
plt.ylabel("equity")
plt.show()


## 10. 泄漏检查清单

在机器学习量化里，漂亮结果经常来自泄漏。每次建模后都问自己：

- 特征是否只用了 t 时刻以前的信息？
- 标签是否用了正确的 `shift(-h)`？
- 标准化、去极值、缺失值参数是否只在训练集上拟合？
- 是否按时间切分，而不是随机切分？
- 是否只在最后看一次测试集？
- 是否考虑了交易成本和信号延迟？

只要其中一条答不上来，结果就不能信。


## 小结

本课完成了 ML 量化的第一个最小闭环：

```text
价格数据 → 特征构造 → 未来收益标签 → 时间切分 → 训练 baseline → 样本外评估
```

这一课的重点不是追求高收益，而是建立正确的建模纪律。下一步可以引入更强的模型，例如逻辑回归、随机森林、XGBoost 或 LightGBM，但前提仍然是：标签、切分、预处理和评估必须干净。

## 验收标准 checklist

- [ ] 能解释为什么量化 ML 不能随机切分数据
- [ ] 能构造未来 5 日收益标签并说明 `shift(-h)` 的含义
- [ ] 能区分特征中的历史信息和标签中的未来信息
- [ ] 能只用训练集拟合标准化参数
- [ ] 能训练一个 baseline 模型并进行样本外评估
- [ ] 能列出至少 3 种常见数据泄漏来源
